# 05 - Évaluation et comparaison

Ce notebook compare les performances du modèle baseline sans contexte et du modèle contextuel sur MovieLens 100K.

L'objectif est de mesurer l'apport des features temporo-sessionnelles via des métriques de ranking (Hit Rate@10, NDCG@10, MRR).


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCF, NCFContext
from metrics import hit_rate, ndcg, mrr

print('Python:', sys.version.split()[0])
print('torch :', torch.__version__)


## Chargement des modèles sauvegardés

Nous cherchons les poids des modèles baseline et contextuel dans `results/models`.


In [ ]:
NOTEBOOK_ROOT = Path('.')
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
BASELINE_PATH = RESULTS_DIR / 'baseline_ncf_ml100k.pt'
CONTEXT_PATH = RESULTS_DIR / 'context_ncf_ml100k.pt'

print('BASELINE_PATH =', BASELINE_PATH)
print('CONTEXT_PATH =', CONTEXT_PATH)

baseline_exists = BASELINE_PATH.exists()
context_exists = CONTEXT_PATH.exists()
print('Baseline exists:', baseline_exists)
print('Context exists:', context_exists)


## Préparation du test set

Nous chargeons MovieLens 100K et appliquons le même preprocessing que dans les notebooks d'entraînement.


In [ ]:
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
df = load_movielens_100k(RAW_ROOT)
df = encode_ids(df)
df = extract_all_context_features(df)
df['label'] = (df['rating'] >= 4).astype(int)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)


## Définitions de métriques et d'évaluation

Fonctions pour calculer les métriques de ranking sur un sous-ensemble d'exemples.


In [ ]:
def evaluate_ranking(model, df, n_items, context_cols=None, k=10, sample_size=500, device=None):
    model.eval()
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    items = torch.arange(n_items, dtype=torch.long, device=device)
    eval_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in eval_df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            user_batch = user_id.expand(n_items)
            if context_cols is None:
                logits = model(user_batch, items).cpu().numpy()
            else:
                context = torch.tensor(row[context_cols].values, dtype=torch.float32, device=device).unsqueeze(0)
                context_batch = context.expand(n_items, len(context_cols))
                logits = model(user_batch, items, context_batch).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(int(item_id in ranking[:k]))
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)


## Chargement des modèles

Nous reconstruisons les architectures et chargeons les poids sauvegardés.


In [ ]:
n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
context_cols = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
context_dim = len(context_cols)

baseline_model = None
context_model = None

if baseline_exists:
    baseline_model = NCF(num_users=n_users, num_items=n_items, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2)
    baseline_model.load_state_dict(torch.load(BASELINE_PATH, map_location='cpu'))
    baseline_model.to('cpu')
    print('Baseline model loaded.')

if context_exists:
    context_model = NCFContext(
        num_users=n_users,
        num_items=n_items,
        context_dim=context_dim,
        embed_dim=32,
        mlp_layers=[64, 32, 16],
        context_embed_dim=32,
        dropout=0.2,
        fusion_type='concat'
    )
    context_model.load_state_dict(torch.load(CONTEXT_PATH, map_location='cpu'))
    context_model.to('cpu')
    print('Context model loaded.')


## Évaluation et tableau de comparaison

Nous calculons les métriques pour chaque modèle et affichons les gains.


In [ ]:
results = []
if baseline_model is not None:
    hit10, ndcg10, mrr_score = evaluate_ranking(baseline_model, test_df, n_items, context_cols=None, k=10, sample_size=500, device='cpu')
    results.append({
        'model': 'baseline',
        'hit10': hit10,
        'ndcg10': ndcg10,
        'mrr': mrr_score
    })

if context_model is not None:
    hit10, ndcg10, mrr_score = evaluate_ranking(context_model, test_df, n_items, context_cols=context_cols, k=10, sample_size=500, device='cpu')
    results.append({
        'model': 'context',
        'hit10': hit10,
        'ndcg10': ndcg10,
        'mrr': mrr_score
    })

results_df = pd.DataFrame(results)
results_df = results_df[['model', 'hit10', 'ndcg10', 'mrr']]
display(results_df)


## Interprétation rapide

Si le modèle contextuel est meilleur, cela valide l'intérêt des features temporelles et de session. Si le score reste similaire, il faudra explorer l'architecture ou la qualité des features.
